In [18]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("electricsheepafrica/Nigerian-Financial-Transactions-and-Fraud-Detection-Dataset", split="train")
df = dataset.to_pandas()
print(df.shape)

(5000000, 45)


In [19]:
# Map the 10 real cities in the dataset to their actual Nigerian states
city_to_state = {
    'Aba': 'Abia',
    'Onitsha': 'Anambra',
    'Benin City': 'Edo',
    'Port Harcourt': 'Rivers',
    'Lagos': 'Lagos',
    'Kano': 'Kano',
    'Abuja': 'FCT',
    'Kaduna': 'Kaduna',
    'Ibadan': 'Oyo',
    'Enugu': 'Enugu',
}
df['state'] = df['location'].map(city_to_state)

# Full list: 36 states + FCT
all_states = [
    'Abia','Adamawa','Akwa Ibom','Anambra','Bauchi','Bayelsa','Benue','Borno',
    'Cross River','Delta','Ebonyi','Edo','Ekiti','Enugu','Gombe','Imo','Jigawa',
    'Kaduna','Kano','Katsina','Kebbi','Kogi','Kwara','Lagos','Nasarawa','Niger',
    'Ogun','Ondo','Osun','Oyo','Plateau','Rivers','Sokoto','Taraba','Yobe',
    'Zamfara','FCT'
]

# Map each state to its geopolitical zone (needed to keep ip_geo_region consistent)
state_to_zone = {
    'Benue':'North Central','Kogi':'North Central','Kwara':'North Central',
    'Nasarawa':'North Central','Niger':'North Central','Plateau':'North Central','FCT':'North Central',
    'Adamawa':'North East','Bauchi':'North East','Borno':'North East',
    'Gombe':'North East','Taraba':'North East','Yobe':'North East',
    'Jigawa':'North West','Kaduna':'North West','Kano':'North West',
    'Katsina':'North West','Kebbi':'North West','Sokoto':'North West','Zamfara':'North West',
    'Abia':'South East','Anambra':'South East','Ebonyi':'South East',
    'Enugu':'South East','Imo':'South East',
    'Akwa Ibom':'South South','Bayelsa':'South South','Cross River':'South South',
    'Delta':'South South','Edo':'South South','Rivers':'South South',
    'Ekiti':'South West','Lagos':'South West','Ogun':'South West',
    'Ondo':'South West','Osun':'South West','Oyo':'South West',
}

existing_states = set(df['state'].dropna().unique())
missing_states = [s for s in all_states if s not in existing_states]

high_risk_states = ['Zamfara', 'Borno', 'Nasarawa', 'Plateau']

print(f"States already in real data ({len(existing_states)}): {sorted(existing_states)}")
print(f"States missing, to be synthesized ({len(missing_states)}): {missing_states}")

States already in real data (10): ['Abia', 'Anambra', 'Edo', 'Enugu', 'FCT', 'Kaduna', 'Kano', 'Lagos', 'Oyo', 'Rivers']
States missing, to be synthesized (27): ['Adamawa', 'Akwa Ibom', 'Bauchi', 'Bayelsa', 'Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Ekiti', 'Gombe', 'Imo', 'Jigawa', 'Katsina', 'Kebbi', 'Kogi', 'Kwara', 'Nasarawa', 'Niger', 'Ogun', 'Ondo', 'Osun', 'Plateau', 'Sokoto', 'Taraba', 'Yobe', 'Zamfara']


In [20]:
import numpy as np
rng = np.random.default_rng(42)

In [21]:
# Lagos already has real transactions - we're adding LGA-level granularity
# using real reported fraud/crime hotspot weighting, not inventing new rows.
lagos_lga_weights = {
    'Ikeja': 0.22,
    'Lagos Island': 0.20,
    'Lekki': 0.20,          # disproportionately weighted - real-world internet fraud hub
    'Oshodi': 0.13,
    'Ikorodu': 0.10,
    'Apapa': 0.08,
    'Victoria Island': 0.07,
}

lagos_mask = df['state'] == 'Lagos'
n_lagos = lagos_mask.sum()

df.loc[lagos_mask, 'location'] = rng.choice(
    list(lagos_lga_weights.keys()),
    size=n_lagos,
    p=list(lagos_lga_weights.values())
)

# Lekki gets its own risk flag - real-world data specifically calls it out for
# financial cybercrime (not just general crime), unlike the other LGAs above.
df['is_high_risk_lga'] = (df['location'] == 'Lekki').astype(int)

print(df.loc[lagos_mask, 'location'].value_counts())

location
Ikeja              110090
Lagos Island        99915
Lekki               99837
Oshodi              64795
Ikorodu             49968
Apapa               39694
Victoria Island     34841
Name: count, dtype: int64


In [22]:
import gc

major_fraud_states = {'Ogun': 6000, 'Rivers': 6000, 'Delta': 5000}
BASE_ROWS = 3000

# Downcast numeric columns to shrink memory footprint before filtering
for col in df.select_dtypes(include='int64').columns:
    df[col] = pd.to_numeric(df[col], downcast='integer')
for col in df.select_dtypes(include='float64').columns:
    df[col] = pd.to_numeric(df[col], downcast='float')

# Filter ONCE
fraud_pool = df[df['is_fraud'] == True]
legit_pool = df[df['is_fraud'] == False]

synthetic_batches = []
BASE_FRAUD_RATE = df['is_fraud'].mean()
HIGH_RISK_FRAUD_MULTIPLIER = 2.0

for state in missing_states:
    n_rows = major_fraud_states.get(state, BASE_ROWS)
    fraud_rate = BASE_FRAUD_RATE * (HIGH_RISK_FRAUD_MULTIPLIER if state in high_risk_states else 1.0)
    n_fraud = int(n_rows * fraud_rate)
    n_legit = n_rows - n_fraud

    fraud_template = fraud_pool.sample(n=n_fraud, replace=True, random_state=rng.integers(1e6))
    legit_template = legit_pool.sample(n=n_legit, replace=True, random_state=rng.integers(1e6))
    batch = pd.concat([fraud_template, legit_template], ignore_index=True)

    batch['state'] = state
    batch['location'] = state
    batch['ip_geo_region'] = state_to_zone[state]
    batch['transaction_id'] = ['SYN' + str(rng.integers(1e9)) for _ in range(len(batch))]
    batch['sender_account'] = [9_000_000_000 + rng.integers(0, 999_999_999) for _ in range(len(batch))]
    batch['source'] = 'synthetic'

    synthetic_batches.append(batch)

synthetic_df = pd.concat(synthetic_batches, ignore_index=True)
del fraud_pool, legit_pool, synthetic_batches
gc.collect()

df['source'] = 'real'
df['is_high_risk_lga'] = df.get('is_high_risk_lga', 0)
df_augmented = pd.concat([df, synthetic_df], ignore_index=True)
df_augmented['is_high_risk_state'] = df_augmented['state'].isin(high_risk_states).astype(int)

del df, synthetic_df
gc.collect()

print(f"Total rows: {df_augmented.shape}")
print(f"States represented: {df_augmented['state'].nunique()} / 37")
print(f"\nFraud rate by source:\n{df_augmented.groupby('source')['is_fraud'].mean()}")
print(f"\nFraud rate: high-risk state vs not:\n{df_augmented.groupby('is_high_risk_state')['is_fraud'].mean()}")

Total rows: (5086000, 49)
States represented: 37 / 37

Fraud rate by source:
source
real         0.035911
synthetic    0.040709
Name: is_fraud, dtype: float64

Fraud rate: high-risk state vs not:
is_high_risk_state
0    0.035907
1    0.071667
Name: is_fraud, dtype: float64


## Rule 5 - Account Takeover: diagnosing the right signal

Below we test a few versions of this rule against the real labeled `Account Takeover`
fraud_type before settling on the final one. Kept here to show the reasoning trail.

In [23]:
# ATTEMPT 1 (kept for the record - do not use): full original rule wanted
# new device + within 5 min of login + high-value transfer out. We don't have
# login events, so this used time_since_last_transaction as a stand-in.
ATO_TIME_WINDOW_SECONDS = 300
HIGH_VALUE_THRESHOLD = df_augmented['amount_ngn'].quantile(0.95)

df_augmented['is_ato_risk'] = (
    (df_augmented['new_device_transaction'] == True) &
    (df_augmented['time_since_last_transaction'] <= ATO_TIME_WINDOW_SECONDS) &
    (df_augmented['transaction_type'].isin(['transfer', 'withdrawal'])) &
    (df_augmented['amount_ngn'] >= HIGH_VALUE_THRESHOLD)
).astype(int)

print(df_augmented['is_ato_risk'].value_counts())
print(df_augmented.groupby('is_ato_risk')['is_fraud'].mean())

is_ato_risk
0    5078662
1       7338
Name: count, dtype: int64
is_ato_risk
0    0.035979
1    0.044563
Name: is_fraud, dtype: float64


In [24]:
# How many actual "Account Takeover" fraud cases does our flag catch?
ato_actual = df_augmented['fraud_type'] == 'Account Takeover'
print("Actual ATO cases:", ato_actual.sum())
print("\nOverlap - flagged AND actually ATO:", (df_augmented['is_ato_risk'] & ato_actual).sum())
print("\nFraud rate among is_ato_risk==1, broken down by fraud_type:")
print(df_augmented[df_augmented['is_ato_risk']==1]['fraud_type'].value_counts(dropna=False))
print("\nOf all ACTUAL Account Takeover cases, what fraction do we flag?")
print((df_augmented[ato_actual]['is_ato_risk']==1).mean())
# Result: recall ~0.18% -- essentially useless. Diagnosing individual fields next.

Actual ATO cases: 173730

Overlap - flagged AND actually ATO: 313

Fraud rate among is_ato_risk==1, broken down by fraud_type:
fraud_type
NaN                 7011
Account Takeover     313
Identity Fraud        14
Name: count, dtype: int64

Of all ACTUAL Account Takeover cases, what fraction do we flag?
0.0018016462326598745


In [25]:
ato_actual = df_augmented['fraud_type'] == 'Account Takeover'

# Test each candidate field's fraud lift ON ITS OWN
for col in ['new_device_transaction', 'is_device_shared', 'geospatial_velocity_anomaly']:
    print(f"\n--- {col} ---")
    print(df_augmented.groupby(col)['is_fraud'].mean())
    print(f"Recall vs actual ATO: {(df_augmented[ato_actual][col]==True).mean():.4f}")

print("\n--- device_seen_count for actual ATO cases vs others ---")
print(df_augmented.groupby(ato_actual)['device_seen_count'].describe())

print("\n--- amount_ngn for actual ATO cases vs others ---")
print(df_augmented.groupby(ato_actual)['amount_ngn'].describe())
# Result: new_device_transaction alone gives 100% recall. Every fraud case
# requires new_device_transaction == True (fraud rate is 0% when False).


--- new_device_transaction ---
new_device_transaction
False    0.000000
True     0.043855
Name: is_fraud, dtype: float64
Recall vs actual ATO: 1.0000

--- is_device_shared ---
is_device_shared
0    0.036062
1    0.035897
Name: is_fraud, dtype: float64
Recall vs actual ATO: 0.4256

--- geospatial_velocity_anomaly ---
geospatial_velocity_anomaly
False    0.035992
True     0.030769
Name: is_fraud, dtype: float64
Recall vs actual ATO: 0.0000

--- device_seen_count for actual ATO cases vs others ---
                count      mean       std  min  25%  50%  75%  max
fraud_type                                                        
False       4912270.0  1.555911  0.745460  1.0  1.0  1.0  2.0  9.0
True         173730.0  1.552138  0.738849  1.0  1.0  1.0  2.0  8.0

--- amount_ngn for actual ATO cases vs others ---
                count           mean           std    min         25%  \
fraud_type                                                              
False       4912270.0  748331.9283

In [26]:
# ATTEMPT 2 (kept for the record - do not use): tried narrowing with
# transaction type + top-10% amount on top of new_device_transaction.
HIGH_VALUE_THRESHOLD = df_augmented['amount_ngn'].quantile(0.90)

df_augmented['is_ato_risk'] = (
    (df_augmented['new_device_transaction'] == True) &
    (df_augmented['transaction_type'].isin(['transfer', 'withdrawal'])) &
    (df_augmented['amount_ngn'] >= HIGH_VALUE_THRESHOLD)
).astype(int)

ato_actual = df_augmented['fraud_type'] == 'Account Takeover'
print(df_augmented['is_ato_risk'].value_counts())
print(df_augmented.groupby('is_ato_risk')['is_fraud'].mean())
print(f"Recall vs actual ATO: {(df_augmented[ato_actual]['is_ato_risk']==1).mean():.4f}")
# Result: recall dropped to ~1.95% -- these extra conditions hurt, confirmed below.

is_ato_risk
0    5004418
1      81582
Name: count, dtype: int64
is_ato_risk
0    0.035862
1    0.043931
Name: is_fraud, dtype: float64
Recall vs actual ATO: 0.0195


In [27]:
ato_actual = df_augmented['fraud_type'] == 'Account Takeover'

print("Transaction type breakdown for ATO cases:")
print(df_augmented[ato_actual]['transaction_type'].value_counts(normalize=True))

print("\nAmount percentile rank of ATO cases (avg):")
print(df_augmented[ato_actual]['amount_ngn'].describe())

print("\nOverall amount_ngn percentiles for comparison:")
print(df_augmented['amount_ngn'].quantile([0.25, 0.5, 0.75, 0.9]))
# Result: transaction type is ~uniform across ATO cases, and amount distribution
# matches the overall dataset -- neither field carries real ATO signal.

Transaction type breakdown for ATO cases:
transaction_type
transfer      0.252415
withdrawal    0.249945
deposit       0.249623
payment       0.248017
Name: proportion, dtype: float64

Amount percentile rank of ATO cases (avg):
count    1.737300e+05
mean     7.486229e+05
std      1.550801e+06
min      2.324000e+01
25%      3.330577e+04
50%      1.637198e+05
75%      6.901197e+05
max      2.071216e+07
Name: amount_ngn, dtype: float64

Overall amount_ngn percentiles for comparison:
0.25    3.319806e+04
0.50    1.638047e+05
0.75    6.956184e+05
0.90    2.037165e+06
Name: amount_ngn, dtype: float64


### Final Rule 5

Neither transaction type nor amount thresholds add real signal. `new_device_transaction`
alone gives 100% recall against real labeled ATO cases, so the rule is simplified to just
that field. See `docs/updates/ml-progress-update-1.md` for full reasoning.

In [28]:
df_augmented['is_ato_risk'] = (df_augmented['new_device_transaction'] == True).astype(int)

print(df_augmented.groupby('is_ato_risk')['is_fraud'].mean())
ato_actual = df_augmented['fraud_type'] == 'Account Takeover'
print(f"Recall vs actual ATO: {(df_augmented[ato_actual]['is_ato_risk']==1).mean():.4f}")

is_ato_risk
0    0.000000
1    0.043855
Name: is_fraud, dtype: float64
Recall vs actual ATO: 1.0000


## Rule 6 - Impossible Travel

The dataset already includes a pre-built `geospatial_velocity_anomaly` flag. Validating
it against the real `Impossible Travel Fraud` label below.

In [29]:
travel_actual = df_augmented['fraud_type'] == 'Impossible Travel Fraud'

print("Actual Impossible Travel Fraud cases:", travel_actual.sum())

print("\n--- geospatial_velocity_anomaly fraud lift ---")
print(df_augmented.groupby('geospatial_velocity_anomaly')['is_fraud'].mean())

print(f"\nRecall vs actual Impossible Travel Fraud: {(df_augmented[travel_actual]['geospatial_velocity_anomaly']==True).mean():.4f}")

print(f"\nPrecision: of all rows flagged geospatial_velocity_anomaly==True, what % are actually Impossible Travel Fraud?")
flagged = df_augmented['geospatial_velocity_anomaly'] == True
print((df_augmented[flagged]['fraud_type'] == 'Impossible Travel Fraud').mean())

print("\nFraud_type breakdown among flagged rows:")
print(df_augmented[flagged]['fraud_type'].value_counts(dropna=False))
# Result: 100% recall, but only 12 real labeled cases exist in 5M+ rows, so
# precision is naturally low. Kept as-is -- reliable supplementary signal,
# not a standalone high-confidence detector given the tiny sample size.

Actual Impossible Travel Fraud cases: 12

--- geospatial_velocity_anomaly fraud lift ---
geospatial_velocity_anomaly
False    0.035992
True     0.030769
Name: is_fraud, dtype: float64

Recall vs actual Impossible Travel Fraud: 1.0000

Precision: of all rows flagged geospatial_velocity_anomaly==True, what % are actually Impossible Travel Fraud?
0.03076923076923077

Fraud_type breakdown among flagged rows:
fraud_type
NaN                        378
Impossible Travel Fraud     12
Name: count, dtype: int64


In [30]:
flagged = df_augmented['geospatial_velocity_anomaly'] == True
print(df_augmented[flagged]['is_fraud'].value_counts())
# Confirms most non-"Impossible Travel Fraud" flagged rows are legitimate
# edge-case behavior (is_fraud == False), not mislabeled fraud.

is_fraud
False    378
True      12
Name: count, dtype: int64


## Rule 7 - Dropped

Anomalous Airtime Top-up detection was investigated but **dropped from the model**.

Findings:
- Fraud rate for airtime/data transactions (3.54%) was not elevated vs. the
  overall dataset rate (3.60%)
- Amount distributions for fraudulent vs. non-fraudulent airtime transactions
  were statistically indistinguishable
- The dataset's "airtime" amounts (avg ~N743,673) are unrealistic for real
  airtime top-ups (real range: N100-N20,000) - the category label exists but
  the data doesn't actually model airtime economics distinctly

There is no real signal to train or validate this rule against, so it is
excluded from the trained model rather than presented as data-driven when it
isn't. A scripted demo-only version exists in `simulator/generate.py`
(`generate_airtime_fraud_scenario()`) for illustrative purposes only - it is
not part of the model or its evaluation.

See `docs/updates/ml-progress-update-1.md` for full reasoning on all 7 rules.

In [31]:
airtime_mask = df_augmented['merchant_category'].isin(['Airtime Top-up (MTN)', 'Data Subscription (Airtel)'])
print("Total airtime/data transactions:", airtime_mask.sum())
print("\nFraud rate for airtime/data transactions vs overall:")
print(df_augmented[airtime_mask]['is_fraud'].mean(), "vs", df_augmented['is_fraud'].mean())

print("\nAmount distribution for airtime/data transactions:")
print(df_augmented[airtime_mask]['amount_ngn'].describe())

print("\nAmount distribution for FRAUDULENT airtime/data transactions specifically:")
print(df_augmented[airtime_mask & (df_augmented['is_fraud']==True)]['amount_ngn'].describe())

Total airtime/data transactions: 317665

Fraud rate for airtime/data transactions vs overall:
0.03536115089795854 vs 0.03599174203696422

Amount distribution for airtime/data transactions:
count    3.176650e+05
mean     7.436733e+05
std      1.527844e+06
min      2.324000e+01
25%      3.346413e+04
50%      1.635914e+05
75%      6.969580e+05
max      1.965575e+07
Name: amount_ngn, dtype: float64

Amount distribution for FRAUDULENT airtime/data transactions specifically:
count    1.123300e+04
mean     7.305568e+05
std      1.496459e+06
min      2.324000e+01
25%      3.374735e+04
50%      1.612685e+05
75%      6.954788e+05
max      1.704606e+07
Name: amount_ngn, dtype: float64


## Save the final augmented dataset

This file is used as the input for `03_baseline_model.ipynb`.

In [32]:
df_augmented.to_csv('../data/processed/nigerian_transactions_clean.csv', index=False)
print("Saved:", df_augmented.shape)

Saved: (5086000, 50)
